In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl
from torch.distributions import Bernoulli
import xarray as xr
import xrft
import numpy as np
import matplotlib.pyplot as plt
import src.data
from scipy.stats import multivariate_normal
import pandas as pd
import functools as ft
from collections import namedtuple
from IPython.display import Markdown, display
from omegaconf import OmegaConf
import yaml
import inspect
import hydra
import os
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib.colors as colors
from matplotlib.colors import LogNorm 
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
from PIL import Image
import kornia.filters as kfilts
from torch.amp import autocast, GradScaler
from scipy.ndimage import gaussian_filter
from scipy.ndimage import binary_dilation
from skimage.morphology import disk, dilation
from sklearn.linear_model import LinearRegression

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = "cpu"

# Utils Functions

In [ ]:
def plot_tensor(tensor, title = 'Logits Plot'):
    pos_tensor_numpy = tensor.numpy()
    plt.imshow(pos_tensor_numpy, cmap='viridis', interpolation='nearest')
    plt.colorbar()
    plt.title(title)
    plt.show()

def rmse_based_scores_from_ds(ds, ref_variable='tgt', study_variable='out'):
    #mask = ~np.isnan(ds['input'])
    try:
        return rmse_based_scores(ds[ref_variable], ds[study_variable])[2:]
    except:
        return [np.nan, np.nan]

def psd_based_scores_from_ds(ds, ref_variable='tgt', study_variable='out'):
    print(ds)
    try:
        return psd_based_scores(ds[ref_variable], ds[study_variable])[1:]
    except:
        return [np.nan, np.nan]

def rmse_based_scores_lead(da_rec, da_ref):
    rmse_t = (
        1.0
        - (((da_rec - da_ref) ** 2).mean(dim=("lon", "lat"))) ** 0.5
        / (((da_ref) ** 2).mean(dim=("lon", "lat"))) ** 0.5
    )
    rmse_xy = (((da_rec - da_ref) ** 2).mean(dim=("time"))) ** 0.5
    rmse_t = rmse_t.rename("rmse_t")
    rmse_xy = rmse_xy.rename("rmse_xy")
    reconstruction_error_stability_metric = rmse_t.std().values
    leaderboard_rmse = (
        1.0 - (((da_rec - da_ref) ** 2).mean()) ** 0.5 / (((da_ref) ** 2).mean()) ** 0.5
    )
    return (
        np.round(leaderboard_rmse.values, 5).item(),
        np.round(reconstruction_error_stability_metric, 5).item(),
    )

def rmse_based_scores_day(da_rec, da_ref):
    # Calculate leaderboard RMSE
    leaderboard_rmse = (
        1.0 - (((da_rec - da_ref) ** 2).mean()) ** 0.5 / (((da_ref) ** 2).mean()) ** 0.5
    )
    return np.round(leaderboard_rmse.values, 5).item()

def rmse_based_scores(da_rec, da_ref):
    rmse_t = (
        1.0
        - (((da_rec - da_ref) ** 2).mean(dim=("lon", "lat"))) ** 0.5
        / (((da_ref) ** 2).mean(dim=("lon", "lat"))) ** 0.5
    )
    rmse_xy = (((da_rec - da_ref) ** 2).mean(dim=("time"))) ** 0.5
    rmse_t = rmse_t.rename("rmse_t")
    rmse_xy = rmse_xy.rename("rmse_xy")
    reconstruction_error_stability_metric = rmse_t.std().values
    leaderboard_rmse = (
        1.0 - (((da_rec - da_ref) ** 2).mean()) ** 0.5 / (((da_ref) ** 2).mean()) ** 0.5
    )
    return (
        rmse_t,
        rmse_xy,
        np.round(leaderboard_rmse.values, 5).item(),
        np.round(reconstruction_error_stability_metric, 5).item(),
    )


def psd_based_scores(da_rec, da_ref):
    print('hello')
    err = da_rec - da_ref
    err["time"] = (err.time - err.time[0]) / np.timedelta64(1, "D")
    signal = da_ref
    signal["time"] = (signal.time - signal.time[0]) / np.timedelta64(1, "D")
    psd_err = xrft.power_spectrum(
        err, dim=["time", "lon"], detrend="constant", window="hann"
    ).compute()
    psd_signal = xrft.power_spectrum(
        signal, dim=["time", "lon"], detrend="constant", window="hann"
    ).compute()

    mean_psd_signal = psd_signal.mean(dim="lat").where(
        (psd_signal.freq_lon > 0.0) & (psd_signal.freq_time > 0), drop=True
    )
    
    mean_psd_err = psd_err.mean(dim="lat").where(
        (psd_err.freq_lon > 0.0) & (psd_err.freq_time > 0), drop=True
    )
    print(mean_psd_err)
    psd_based_score = 1.0 - mean_psd_err / mean_psd_signal
    level = [0.5]
    cs = plt.contour(
        1.0 / psd_based_score.freq_lon.values,
        1.0 / psd_based_score.freq_time.values,
        psd_based_score,
        level,
    )
    x05, y05 = cs.collections[0].get_paths()[0].vertices.T
    plt.close()

    shortest_spatial_wavelength_resolved = np.min(x05)
    shortest_temporal_wavelength_resolved = np.min(y05)
    psd_da = 1.0 - mean_psd_err / mean_psd_signal
    psd_da.name = "psd_score"
    return (
        psd_da.to_dataset(),
        np.round(shortest_spatial_wavelength_resolved, 3).item(),
        np.round(shortest_temporal_wavelength_resolved, 3).item(),
    )

def weighted_mse(err, weight = None):
    if weight is None:
        err_w = err
        non_zeros = (torch.ones_like(err) == 0.0)
    else:
        err_w = err * weight[None, ...]
        non_zeros = (torch.ones_like(err) * weight[None, ...]) == 0.0
    err_num = err.isfinite() & ~non_zeros
    if err_num.sum() == 0:
        return torch.scalar_tensor(1000.0, device=err_num.device).requires_grad_()
    loss = F.mse_loss(err_w[err_num], torch.zeros_like(err_w[err_num]))
    return loss

def pprint_cfg(cfg):
    display(Markdown("""```yaml\n\n""" +yaml.dump(OmegaConf.to_container(cfg), default_flow_style=None, indent=2)+"""\n\n```"""))


def update_dz_fig(frame, da, da_4dVN):
    plt.clf()  # Clear the previous frame
    global_min = np.min([da[frame, :, :], da_4dVN[frame, :, :]])
    global_max = np.max([da[frame, :, :], da_4dVN[frame, :, :]])
    
    mu_4dVN = rmse_based_scores_day(da[frame, :, :], da_4dVN[frame, :, :])

    corr_da_4dVN = xr.corr(da[frame, :, :], da_4dVN[frame, :, :])

    # Ground truth
    ax0 = fig.add_subplot(gs[0])
    im0 = ax0.imshow(da[frame, :, :] + 1, extent=[-60, -54, 32, 38], cmap='viridis', vmin=global_min, vmax=global_max)
    ax0.text(0.5, 1.05, 'Ground truth', ha='center', va='bottom', transform=ax0.transAxes, fontsize=14)

    title_4dVarNet = '4dVarNet'
    subtitle_4dVarNet = f'Correlation: {corr_da_4dVN:.2f}\nRMSE Score: {mu_4dVN:.2f}'

    # 4dVarNet
    ax2 = fig.add_subplot(gs[1])
    im2 = ax2.imshow(da_4dVN[frame, :, :] + 1, extent=[-60, -54, 32, 38], cmap='viridis', vmin=global_min, vmax=global_max)
    ax2.text(0.5, 1.1, title_4dVarNet, ha='center', va='bottom', transform=ax2.transAxes, fontsize=14)
    ax2.text(0.5, 1.01, subtitle_4dVarNet, ha='center', va='bottom', transform=ax2.transAxes, fontsize=10, linespacing=1.5)

    # Colorbar
    ax3 = fig.add_subplot(gs[2])
    plt.colorbar(im2, cax=ax3, label='ECS')

    ax3_position = ax3.get_position()
    new_position = [ax3_position.x0 - 0.03, ax3_position.y0, ax3_position.width, ax3_position.height]
    ax3.set_position(new_position)

# Field Selection

In [ ]:
# tgt_ds_ecs = xr.open_dataset('/Odyssey/public/natl60/celerity/NATL60GULF-CJM165_cutoff_freq_regrid_0_1000m.nc')
# tgt_ds = xr.open_dataset('natl_gf_w_5nadirs.nc')
# time_step = '2012-11-13T12:00:00.000000000'  # Example time step
# inp_da = tgt_ds.ssh.sel(time=time_step)
# #inp_da = tgt_ds_ecs.ecs.sel(time=time_step)
# inp_da.plot()
# plt.title(f"Sample Values at Time Step {time_step}")
# plt.show()

# lat_shape = len(inp_da.lat)
# lon_shape = len(inp_da.lon)

# crop_lat_start =  20
# crop_lat_end = lat_shape -  20
# crop_lon_start =  20
# crop_lon_end = lon_shape -  20

# inp_da_GS = inp_da.isel(lat=slice(crop_lat_start, crop_lat_end),
#                      lon=slice(crop_lon_start, crop_lon_end))
# lambda_value = 15.0

In [ ]:
tgt_ds_ecs = xr.open_dataset('/Odyssey/public/natl60/celerity/NATL60GULF-CJM165_cutoff_freq_regrid_0_1000m.nc')
tgt_ds_dim = xr.open_dataset('natl_gf_w_5nadirs.nc')
tgt_ds = xr.open_dataset('/Odyssey/public/natl60/ssh/NATL60-CJM165-ssh-2012-2013-1_20.nc')
time_step_ssh = '2012-11-13T00:00:00.000000000'
time_step     = '2012-11-13T12:00:00.000000000' 
# Example time step
inp_da = tgt_ds.ssh.sel(time=time_step_ssh)
inp_da_dim = tgt_ds_dim.ssh.sel(time=time_step)
#inp_da = tgt_ds_ecs.ecs.sel(time=time_step)

# lat_shape = len(inp_da_dim.lat)
# lon_shape = len(inp_da_dim.lon)

# crop_lat_start =  20
# crop_lat_end = lat_shape -  20
# crop_lon_start =  20
# crop_lon_end = lon_shape -  20

inp_da_GS = inp_da.sel(lat=slice(30., 44.95), 
                       lon=slice(-65.95, -50.))

lat_shape = len(inp_da_GS.lat)
lon_shape = len(inp_da_GS.lon)

crop_lat_start =  20
crop_lat_end = lat_shape -  20
crop_lon_start =  20
crop_lon_end = lon_shape -  20

inp_da_GS.fillna(0.0)
lambda_value = 15.0
inp_da_GS.plot()
plt.title(f"Sample Values at Time Step {time_step}")
plt.show()

# OI Code

In [ ]:
def optimal_interpolation(data_with_nans, length_scale=15., sigma_f=1.0, sigma_n=0.1):
    """
    Perform optimal interpolation (kriging) on a 2D tensor with missing data (NaNs).

    Parameters:
    - data_with_nans (torch.Tensor): 2D tensor with missing values represented as NaN.
    - length_scale (float or torch.Tensor): Length scale parameter for the RBF kernel.
    - sigma_f (float or torch.Tensor): Signal variance parameter for the RBF kernel.
    - sigma_n (float or torch.Tensor): Noise variance parameter.

    Returns:
    - reconstructed_data (torch.Tensor): 2D tensor with missing values reconstructed.
    """
    device = data_with_nans.device
    H, W = data_with_nans.shape

    observed_indices = ~torch.isnan(data_with_nans)
    observed_positions = torch.nonzero(observed_indices, as_tuple=False).float()  # Positions (i, j)
    observed_values = data_with_nans[observed_indices].to(device)

    missing_indices = torch.isnan(data_with_nans)
    missing_positions = torch.nonzero(missing_indices, as_tuple=False).float()

    observed_positions_x = observed_positions[:, 1] #/ (W - 1) * 10  
    observed_positions_y = observed_positions[:, 0] #/ (H - 1) * 10  
    observed_coords = torch.stack([observed_positions_x, observed_positions_y], dim=1)

    missing_positions_x = missing_positions[:, 1] #/ (W - 1) * 10
    missing_positions_y = missing_positions[:, 0] #/ (H - 1) * 10
    missing_coords = torch.stack([missing_positions_x, missing_positions_y], dim=1)

    # Define the covariance function (RBF kernel)
    def cov_func(x1, x2, length_scale, sigma_f):
        """
        Compute the covariance matrix using the RBF kernel.
        """
        x1 = x1.unsqueeze(1)  # Shape: [N1, 1, 2]
        x2 = x2.unsqueeze(0)  # Shape: [1, N2, 2]
        sqdist = ((x1 - x2) ** 2).sum(dim=2)  # Shape: [N1, N2]
        return sigma_f ** 2 * torch.exp(-0.5 / length_scale ** 2 * sqdist)

    # Ensure hyperparameters are tensors on the correct device
    if not isinstance(length_scale, torch.Tensor):
        length_scale = torch.tensor(length_scale, device=device)
    if not isinstance(sigma_f, torch.Tensor):
        sigma_f = torch.tensor(sigma_f, device=device)
    if not isinstance(sigma_n, torch.Tensor):
        sigma_n = torch.tensor(sigma_n, device=device)

    # Compute covariance matrices
    K = cov_func(observed_coords, observed_coords, length_scale, sigma_f)
    K += sigma_n ** 2 * torch.eye(K.size(0), device=device)  # Add noise variance
    K_s = cov_func(observed_coords, missing_coords, length_scale, sigma_f)

    # Cholesky decomposition
    L = torch.cholesky(K + 1e-6 * torch.eye(K.size(0), device=device))  # Add jitter for numerical stability

    # Solve for alpha
    alpha = torch.cholesky_solve(observed_values.unsqueeze(1), L)

    # Predict mean at missing points
    mean_pred = K_s.t().matmul(alpha).squeeze()
    mean_pred = mean_pred.to(torch.float32)

    # Reconstruct the data tensor
    reconstructed_data = data_with_nans.clone()
    reconstructed_data[missing_indices] = mean_pred

    return reconstructed_data

# Warped Field Perturbations

In [ ]:
N = lat_shape  # Grid size
L = 5.0  # Spatial correlation length
T_corr = 1.0  # Temporal correlation length
sigma = 60  # Standard deviation of the field

def generate_correlated_fields(N, L, T_corr, sigma, num_fields=10, device=device,seed=41):
    """
    Generate a series of 2D fields with both spatial and temporal correlations.

    Parameters:
        N (int): Grid size (assumed to be square, NxN).
        L (float): Spatial correlation length.
        T_corr (float): Temporal correlation length.
        sigma (float): Standard deviation of the field.
        num_fields (int): Number of fields to generate (default is 10).
        device (str): Device to run the computations on ('cpu' or 'cuda').

    Returns:
        torch.Tensor: Generated fields of shape (num_fields, N, N).
    """
    torch.manual_seed(seed)
    # Define the time points for the fields
    time_points = torch.linspace(0, num_fields - 1, num_fields, device=device)
    # Compute the temporal covariance matrix
    C_temporal = torch.exp(-abs(time_points[:, None] - time_points[None, :]) / T_corr)
    # Perform Cholesky decomposition to get the temporal correlation factors
    L_chol = torch.linalg.cholesky(C_temporal)
    # Generate independent Gaussian white noise for each time point
    white_noises = torch.randn((num_fields, N, N), device=device)
    # Combine white noise using the Cholesky factors to induce temporal correlation
    temporal_correlated_noises = torch.matmul(L_chol, white_noises.view(num_fields, -1)).view(num_fields, N, N)
    # Generate 2D grid of wavenumbers for spatial correlation
    kx = torch.fft.fftfreq(N, device=device) * N
    ky = torch.fft.fftfreq(N, device=device) * N
    k = torch.sqrt(kx[:, None]**2 + ky[None, :]**2)
    cutoff_mask = (k < 200).float()  # High-frequency cutoff
    # apply - if we apply the same approach to vorticity, and then obtain 
    # stream function, 
    # Spatial covariance (Power spectrum) for Gaussian covariance
    P_k = torch.exp(-0.5 * (k * L)**3)
    P_k[0, 0] = 0.0
    P_k = P_k / torch.sum(P_k)
    # Generate fields using Fourier transform
    fields = []
    for i in range(num_fields):
        noise_ft = torch.fft.fft2(temporal_correlated_noises[i])
        field_ft = noise_ft * sigma**2 * torch.sqrt(P_k) * cutoff_mask
        field = torch.fft.ifft2(field_ft).real
        fields.append(field)
    return torch.stack(fields)

# Generate a Gaussian random field with a high-frequency cutoff
displacement_x = generate_correlated_fields(N, L, T_corr, sigma, device=device, seed=41)
displacement_y = generate_correlated_fields(N, L, T_corr, sigma, device=device, seed=41)
# fake_dx=torch.zeros_like(displacement_x)

In [ ]:
# We now perturb the fields at the same way,
def warp_field(field, dx, dy):
    """
    Warp a 2D field based on displacement fields dx and dy.
    field (torch.Tensor): Input field of shape (batch_size, height, width)
    dx (torch.Tensor): X-displacement field of shape (batch_size, height, width)
    dy (torch.Tensor): Y-displacement field of shape (batch_size, height, width)
    """
    batch_size, _, height, width = field.shape
    
    # Create base grid
    y, x = torch.meshgrid(torch.arange(height), torch.arange(width), indexing='ij')
    base_grid = torch.stack((x, y), dim=-1).float()
    dx=dx.to(field.dtype)  # Adjust size as needed
    dy=dy.to(field.dtype) 
    # Add batch dimension and move to the same device as input field
    base_grid = base_grid.unsqueeze(0).repeat(batch_size,1,1,1).to(field.device)

    # Apply displacements
    sample_grid = base_grid + torch.stack((dx, dy), dim=-1)
    sample_grid[..., 0] = sample_grid[..., 0] % (width)
    sample_grid[..., 1] = sample_grid[..., 1] % (height)
    

    # Normalize grid to [-1, 1] range
    sample_grid[..., 0] = 2 * sample_grid[..., 0] / (width) - 1
    sample_grid[..., 1] = 2 * sample_grid[..., 1] / (height) - 1

    # Perform sampling
    warped_field = F.grid_sample(field, sample_grid, mode='bilinear', padding_mode='reflection', align_corners=False)
    return warped_field,torch.max(sample_grid), torch.min(sample_grid)

In [ ]:
batch_size = 10
channels = 1

# Prepare data
mean_tgt = inp_da.mean().item()
std_tgt = inp_da.std().item()

tens_inp = torch.from_numpy(inp_da.values).float().to(device)
tens_inp = (tens_inp - mean_tgt) / std_tgt
tens_inp.unsqueeze_(0)

tens_inp = tens_inp.repeat(batch_size, 1, 1, 1)
# Create a sample field (e.g., a gradient)
field = tens_inp
displacement_x = displacement_x 
displacement_y = displacement_y
mean_x=torch.tensor(0.0)
mean_y=torch.tensor(0.0)
#mean_x=torch.tensor(6.0)
#mean_y=torch.tensor(5.0)

# Perform warping
warped_field,max_grid,min_grid = warp_field(field, displacement_x+mean_x, displacement_y+mean_y)

print(f"Input field shape: {field.shape}")
print(f"Warped field shape: {warped_field.shape}")

In [ ]:
# Function to create a 2D Gaussian kernel
def gaussian_kernel(kernel_size=5, sigma=1.0, dtype=torch.float32, device='cuda'):
    # Create a 2D grid of coordinates (x, y)
    x = torch.arange(kernel_size, dtype=dtype, device=device) - (kernel_size - 1) / 2
    y = torch.arange(kernel_size, dtype=dtype, device=device) - (kernel_size - 1) / 2
    xx, yy = torch.meshgrid(x, y)

    # Compute the Gaussian function on the grid
    kernel = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))

    # Normalize the kernel to ensure the sum of the weights is 1
    kernel = kernel / kernel.sum()
    return kernel

# Function to apply Gaussian smoothing to a batch of images
def apply_gaussian_smoothing(input_tensor, kernel_size=15, sigma=1.0):
    dtype = input_tensor.dtype
    device = input_tensor.device
    kernel = gaussian_kernel(kernel_size, sigma, dtype=dtype, device=device)
    
    # Reshape the kernel to be [1, 1, kernel_size, kernel_size] (for 2D convolution)
    kernel = kernel.view(1, 1, kernel_size, kernel_size)

    # Expand the kernel to apply it across all channels and the batch
    kernel = kernel.expand(1, 1, kernel_size, kernel_size)

    padding=kernel_size//2
    # Pad the input tensor with circular padding
    padded_input = F.pad(input_tensor, (padding, padding, padding, padding), mode='reflect')

    # Apply the convolution (use groups=batch_size to apply the kernel individually per batch element)
    smoothed_tensor = F.conv2d(padded_input, kernel, groups=1)

    # Remove the extra channel dimension
    return smoothed_tensor 

In [ ]:
# Smmothing
smoothed_tensor=apply_gaussian_smoothing(warped_field,kernel_size=10,sigma=1.5)

In [ ]:
# We now generate a spatially correlated gaussian random field that we can
plt.figure(figsize=(16, 5))

for i in range(5):
    plt.subplot(2, 5, i + 1)
    plt.imshow(displacement_x[i].cpu().numpy(), cmap='PuOr')
    plt.colorbar()
    plt.title(f"grf at t={i+1}")

for i in range(5, 10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(displacement_x[i].cpu().numpy(), cmap='PuOr')
    plt.colorbar()
    plt.title(f"grf at t={i+1}")

plt.show()

In [ ]:
border_size = 20

plt.figure(figsize=(22,14))
for i in range(4):
    plt.subplot(4,4,1+i)
    plt.imshow(field[i+4,0,border_size:-border_size,border_size:-border_size].cpu().numpy(), cmap='RdBu')
    plt.colorbar()
    plt.title(f'true field at t={i}')

    plt.subplot(4,4,5+i)
    plt.imshow(warped_field[i+4,0,border_size:-border_size,border_size:-border_size].cpu().numpy(), cmap='RdBu')
    plt.colorbar()
    plt.title(f'perturbed field at t={i}')

    plt.subplot(4,4,9+i)
    plt.imshow(smoothed_tensor[i+4,0,border_size:-border_size,border_size:-border_size].cpu().numpy(), cmap='RdBu')
    plt.colorbar()
    plt.title(f'smoothed field x at t={i}')

    plt.subplot(4,4,13+i)
    plt.imshow(np.abs(warped_field[i+4,0,border_size:-border_size,border_size:-border_size].cpu().numpy() - field[i+4,0,border_size:-border_size,border_size:-border_size].cpu().numpy()), cmap='RdBu')
    plt.colorbar()
    plt.title(f'error field x at t={i}')

# GS Model

In [ ]:
class SmoothingConv(nn.Module):
    def __init__(self):
        super().__init__()
        # A simple 3x3 average kernel with padding=1
        kernel = torch.ones(1, 1, 3, 3) / 9.0
        self.register_buffer('kernel', kernel)

    def forward(self, x):
        # x should be shape [N, 1, lat, lon] for this to work straightforwardly
        return F.conv2d(x, self.kernel, padding=1)

smoother = SmoothingConv().to(device)

def smoothing_conv_loss(logits):
    # Suppose logits is [2, time, lat, lon]. We can choose channel=1 if we only
    # want to smooth the "selected" channel. We reshape to [N= time, C=1, lat, lon].
    # Or simply do it for each (time, channel) separately in a loop.
    
    # Example: smooth only the second channel (where index=1).
    to_smooth = logits[1]  # shape: [time, lat, lon]
    to_smooth = to_smooth.unsqueeze(1)  # [time, 1, lat, lon]
    
    smoothed = smoother(to_smooth)
    # L2 difference between the original and the smoothed version
    penalty = (to_smooth - smoothed).pow(2).mean()
    return penalty

def spatial_smoothness_loss(self, logits):
    """
    Compute a simple spatial smoothness penalty by penalizing large 
    differences between neighboring logits along the spatial dims.
    """
    # logits is shape [2, time, lat, lon]
    # Usually, we penalize only across lat/lon. 
    # time dimension can be included or excluded, depending on use-case.
    grad_x = logits[:, :, :, 1:] - logits[:, :, :, :-1]  # difference along lon
    grad_y = logits[:, :, 1:, :] - logits[:, :, :-1, :]  # difference along lat
    # L2 penalty on these gradients
    loss_x = (grad_x ** 2).mean()
    loss_y = (grad_y ** 2).mean()
    return loss_x + loss_y
    
def laplacian_loss(self, logits):
    """
    Penalize the 2D Laplacian to enforce second-order smoothness.
    """
    # logits: [2, time, lat, lon]
    # focusing on the lat-lon dimension here.
    lap = (
          logits[:, :, 2:, 1:-1]
        + logits[:, :, :-2, 1:-1]
        + logits[:, :, 1:-1, 2:]
        + logits[:, :, 1:-1, :-2]
        - 4 * logits[:, :, 1:-1, 1:-1]
    )
    return (lap ** 2).mean()

In [ ]:
class GSModel(nn.Module):
    def __init__(self, time, lat, lon, rate, device):
        super(GSModel, self).__init__()
        self.logits = nn.Parameter(torch.zeros((2, time, lat, lon)))
        #self.logits = torch.zeros((2, time, lat, lon))
        self.logits.data[1, :, :, :] = np.log(rate / (1 - rate))
        self.time_window, self.lat, self.lon  = time, lat, lon
        self.device = device
        #self.length_scale = torch.tensor(22.0) 
        self.length_scale = nn.Parameter(torch.tensor(25.0))
        
    def gumbel_softmax(self, logits, tau=1.0, hard=False):
        gumbels = -torch.empty_like(logits, memory_format=torch.legacy_contiguous_format).exponential_().log()  # ~Gumbel(0,1)
        gumbels = (logits + gumbels) / tau  # ~Gumbel(logits,tau)
        y_soft = gumbels.softmax(dim=0)
        if hard:
            index = y_soft.max(dim=0, keepdim=True)[1]
            y_hard = torch.zeros_like(logits, memory_format=torch.legacy_contiguous_format).scatter_(dim=0, index=index, value=1.0)
            ret = y_hard - y_soft.detach() + y_soft
        else:
            ret = y_soft
        return ret
    
    def spatial_smoothness_loss(logits):
        """
        Compute a simple spatial smoothness penalty by penalizing large 
        differences between neighboring logits along the spatial dims.
        """
        # logits is shape [2, time, lat, lon]
        # Usually, we penalize only across lat/lon. 
        # time dimension can be included or excluded, depending on use-case.

        grad_x = logits[:, :, :, 1:] - logits[:, :, :, :-1]  # difference along lon
        grad_y = logits[:, :, 1:, :] - logits[:, :, :-1, :]  # difference along lat

        # L2 penalty on these gradients
        loss_x = (grad_x ** 2).mean()
        loss_y = (grad_y ** 2).mean()

        return loss_x + loss_y

    def compute_logits_loss(self, logits_init, batch, budget_obs = 0.01, tau=1.0, weight=1.0, hard=True):
        tgt_inp = batch.tgt.clone()
        logits = logits_init.to(self.device)
    
        # Generate mask using Gumbel-Softmax
        gs_output = F.gumbel_softmax(logits, tau=tau, hard=hard, dim=0)[0, :, :, :]
        mask_input = gs_output.view(self.time_window,self.lat, self.lon)
        mask_input = mask_input.unsqueeze(0)  # Add batch dimension if necessary
        normalized_selected_points = mask_input.mean()
        # Create mask for observed data
        mask = mask_input
        # Apply mask to target input
        selected_tgt_inp = tgt_inp * mask
        selected_tgt_inp[mask == 0] = float('nan')
        # Perform optimal interpolation
        #output = optimal_interpolation(selected_tgt_inp[0,0])
        output = optimal_interpolation(selected_tgt_inp[0,0], length_scale=self.length_scale)
        #mean_tgt = output.mean().item()
        #std_tgt = output.std().item()
        output = output.unsqueeze(0)
        output = output.unsqueeze(0)

        # Ensure the data types match
        output = output.to(tgt_inp.dtype)

        # Compute loss
        #loss_mse = weighted_mse(output - batch.tgt, lit_mod.rec_weight)
        # Define the padding size
        padding_size = 10  # Adjust this value as needed
        # Create a mask for the center region
        center_mask = torch.zeros_like(output)
        center_mask[:,:, padding_size:-padding_size, padding_size:-padding_size] = 1

        # Apply the center mask to the output and target
        output_center = output #* center_mask
        tgt_center = batch.tgt #* center_mask
        #output_center = (output_center - mean_tgt) / std_tgt
        #tgt_center = (tgt_center - mean_tgt) / std_tgt
        # Compute loss only for the center region
        loss_mse = F.mse_loss(output_center, tgt_center)

        # Compute normalized selected points
        valid_points_mask = ~torch.isnan(selected_tgt_inp)
        num_non_nan_elements = valid_points_mask.float().sum()
        total_points = mask.numel()
        #normalized_selected_points = num_non_nan_elements / total_points
        if normalized_selected_points >= budget_obs:
            weight = weight
        else:
            weight = 0
        loss_smoothness = self.spatial_smoothness_loss(logits)
        loss = loss_mse + weight*normalized_selected_points + 0.01*loss_smoothness
        return output, loss, mask_input, normalized_selected_points, loss_mse

    def normalize_logits(self):
        norm = self.logits.norm(p=2, dim=0, keepdim=True)
        self.logits.data = self.logits.data / norm

    def forward(self, batch, budget_obs, tau, weight, n_draw):
        total_loss = 0
        total_points = 0
        total_mse = 0
        for _ in range(n_draw):
            output, loss_draw, mask, normalized_selected_points, loss_mse = self.compute_logits_loss(self.logits, batch, budget_obs=budget_obs, tau=tau, weight=weight, hard=True)
            total_loss += loss_draw
            total_points += normalized_selected_points
            total_mse += loss_mse
            del output, mask
        mean_loss = total_loss / n_draw
        mean_points = total_points / n_draw
        mean_mse = total_mse / n_draw
        #print(self.length_scale)
        return mean_loss, mean_points, mean_mse, self.length_scale

In [ ]:
class GSModel_warped(nn.Module):
    def __init__(self, time, lat, lon, rate, device):
        super(GSModel_warped, self).__init__()
        self.logits = nn.Parameter(torch.zeros((2, time, lat, lon)))
        #self.logits = torch.zeros((2, time, lat, lon))
        self.logits.data[1, :, :, :] = np.log(rate / (1 - rate))
        self.time_window, self.lat, self.lon  = time, lat, lon
        self.device = device
        self.length_scale = nn.Parameter(torch.tensor(25.0))
    
    def spatial_smoothness_loss(self, logits):
        """
        Compute a simple spatial smoothness penalty by penalizing large 
        differences between neighboring logits along the spatial dims.
        """
        # logits is shape [2, time, lat, lon]
        # Usually, we penalize only across lat/lon. 
        # time dimension can be included or excluded, depending on use-case.

        grad_x = logits[:, :, :, 1:] - logits[:, :, :, :-1]  # difference along lon
        grad_y = logits[:, :, 1:, :] - logits[:, :, :-1, :]  # difference along lat

        # L2 penalty on these gradients
        loss_x = (grad_x ** 2).mean()
        loss_y = (grad_y ** 2).mean()

        return loss_x + loss_y
    
    def laplacian_loss(self, logits):
        """
        Penalize the 2D Laplacian to enforce second-order smoothness.
        """
        # logits: [2, time, lat, lon]
        # focusing on the lat-lon dimension here.
        lap = (
              logits[:, :, 2:, 1:-1]
            + logits[:, :, :-2, 1:-1]
            + logits[:, :, 1:-1, 2:]
            + logits[:, :, 1:-1, :-2]
            - 4 * logits[:, :, 1:-1, 1:-1]
        )
        return (lap ** 2).mean()
    
    def gumbel_softmax(self, logits, tau=1.0, hard=False):
        gumbels = -torch.empty_like(logits, memory_format=torch.legacy_contiguous_format).exponential_().log()  # ~Gumbel(0,1)
        gumbels = (logits + gumbels) / tau  # ~Gumbel(logits,tau)
        y_soft = gumbels.softmax(dim=0)
        if hard:
            index = y_soft.max(dim=0, keepdim=True)[1]
            y_hard = torch.zeros_like(logits, memory_format=torch.legacy_contiguous_format).scatter_(dim=0, index=index, value=1.0)
            ret = y_hard - y_soft.detach() + y_soft
        else:
            ret = y_soft
        return ret

    def compute_logits_loss(self, batch, budget_obs = 0.01, tau=1.0, weight=1.0, hard=True):
        logits = self.logits.to(self.device)
        batch_size = batch.tgt.size(0)
        total_loss = 0
        total_normalized_selected_points = 0
        total_loss_mse = 0
        avg_logits = torch.zeros_like(logits)
        for i in range(batch_size):
            tgt_inp = batch.tgt[i].clone()
            #logits = self.logits.to(self.device)
            # Generate mask using Gumbel-Softmax
            gs_output = F.gumbel_softmax(logits, tau=tau, hard=hard, dim=0)[0, :, :, :]
            mask_input = gs_output.view(self.time_window, self.lat, self.lon)
            mask_input = mask_input.unsqueeze(0)  # Add batch dimension if necessary
            normalized_selected_points = mask_input.mean()

            # Create mask for observed data
            mask = mask_input
            # Apply mask to target input
            selected_tgt_inp = tgt_inp * mask
            selected_tgt_inp[mask == 0] = float('nan')
            # Perform optimal interpolation
            output = optimal_interpolation(selected_tgt_inp[0,0], length_scale=self.length_scale)
            output = output.unsqueeze(0)
            output = output.unsqueeze(0)

            # Ensure the data types match
            output = output.to(tgt_inp.dtype)

            # Compute loss
            # Define the padding size
            padding_size = 10  # Adjust this value as needed
            # Create a mask for the center region
            center_mask = torch.zeros_like(output)
            center_mask[:,:, padding_size:-padding_size, padding_size:-padding_size] = 1

            # Apply the center mask to the output and target
            output_center = output #* center_mask
            tgt_center = batch.tgt[i] #* center_mask

            # Compute loss only for the center region
            loss_mse = F.mse_loss(output_center, tgt_center)

            #normalized_selected_points = num_non_nan_elements / total_points
            if normalized_selected_points >= budget_obs:
                weight = weight
            else:
                weight = 0
            #loss = loss_mse + weight*normalized_selected_points
            #loss_smoothness = self.laplacian_loss(logits)
            #loss = loss_mse + weight*normalized_selected_points + 0.1*loss_smoothness
            loss = loss_mse + weight*normalized_selected_points

            total_loss += loss
            total_normalized_selected_points += normalized_selected_points
            total_loss_mse += loss_mse
            # Convert logits to probabilities, average them, and convert back to logits
        #     probs = torch.sigmoid(logits)
        #     avg_logits += probs

        # avg_logits /= batch_size
        # avg_logits = torch.log(avg_logits / (1 - avg_logits))
        # avg_logits = nn.Parameter(avg_logits)  # Wrap in nn.Parameter
        # self.logits = avg_logits

        avg_loss = total_loss / batch_size
        avg_normalized_selected_points = total_normalized_selected_points / batch_size
        avg_loss_mse = total_loss_mse / batch_size
        
        return output, avg_loss, mask_input, avg_normalized_selected_points, avg_loss_mse

    def normalize_logits(self):
        norm = self.logits.norm(p=2, dim=0, keepdim=True)
        self.logits.data = self.logits.data / norm

    def forward(self, batch, budget_obs, tau, weight, n_draw):
        total_loss = 0
        total_points = 0
        total_mse = 0
        for _ in range(n_draw):
            output, loss_draw, mask, normalized_selected_points, loss_mse = self.compute_logits_loss(batch, budget_obs=budget_obs, tau=tau, weight=weight, hard=True)
            total_loss += loss_draw
            total_points += normalized_selected_points
            total_mse += loss_mse
            del output, mask
        mean_loss = total_loss / n_draw
        mean_points = total_points / n_draw
        mean_mse = total_mse / n_draw
        #print(self.length_scale)
        return mean_loss, mean_points, mean_mse, self.length_scale

# Optimization loop

In [ ]:
print(f"Warped field shape: {warped_field.shape}")
inp_da_GS_warped = warped_field[:, 0, 20:220, 20:220]

batch_size = 2
time = 1
lat = 200
lon =200

n_draw = 15
n_iter = 1000
rate = 0.999
budget_obs = 1 - 0.999
accumulation_steps = 4

warmup_iterations = 100
delta_weight = 0.2
max_weight = 20.0
TrainingItem = namedtuple('TrainingItem', ['input', 'tgt'])
smoothness_weight = 0.001
# Initialize model
model_GS = GSModel_warped(time, lat, lon, rate, device).to(device)

inputs = []
targets = []

for j in range(batch_size):
    inp = inp_da_GS_warped[j]
    # Prepare data
    mean_tgt = inp.mean().item()
    std_tgt = inp.std().item()

    tens_inp_da_warped = inp
    tens_inp_da_warped = (tens_inp_da_warped - mean_tgt) / std_tgt
    tens_inp_da_warped = tens_inp_da_warped.unsqueeze(0)
    tens_inp_da_warped = tens_inp_da_warped.unsqueeze(0)
    tens_inp_da_warped.requires_grad_(True)
    
    inputs.append(tens_inp_da_warped)
    targets.append(tens_inp_da_warped)

# Stack inputs and targets to create a batch
inputs = torch.stack(inputs)
targets = torch.stack(targets)
batch = TrainingItem(input=inputs, tgt=targets)
print(batch.tgt.size())
start_temp = 5.0
min_temp = 0.1
alpha = 0.9
lr = 1e-1
temperature = start_temp
total_loss_list = []
mean_loss_list = []
mean_points_list = []
mean_mse_list = []
length_scale_list = []

decay_rate = (min_temp / start_temp) ** (1 / (n_iter - 1))

# Initialize optimizer
optimizer = optim.Adam(model_GS.parameters(), lr=lr)
scaler = GradScaler()

# Define pooling layer
pooling_layer = nn.AvgPool2d(kernel_size=3, stride=1, padding=1)

# Training loop
for k in range(n_iter):
    optimizer.zero_grad()
    total_points = 0
    total_mse = 0
    # weight = (1/1)*(k/n_iter)
    # Adjust weight based on the current iteration
    if k < warmup_iterations:
        weight = 0
    else:
        # Calculate weight incrementally after warmup
        weight = min((k - warmup_iterations + 1) * delta_weight, max_weight)
    for i in range(accumulation_steps):
        with autocast('cuda'):  # Mixed precision training
            mean_loss, normalized_selected_points, loss_mse, length_scale = model_GS(batch, budget_obs=budget_obs, tau=temperature, n_draw=n_draw, weight=weight)
            mean_loss = mean_loss / accumulation_steps  # Normalize loss by accumulation steps
            total_points += normalized_selected_points / accumulation_steps  # Accumulate normalized points
            total_mse += loss_mse / accumulation_steps  # Accumulate MSE loss
            sm_penalty = smoothing_conv_loss(model_GS.logits)
            total_loss = mean_loss + smoothness_weight * sm_penalty
        scaler.scale(total_loss).backward()
    
    scaler.step(optimizer)
    scaler.update()
    total_loss_list.append(total_loss.item())
    mean_loss_list.append(mean_loss.item())
    mean_points_list.append(total_points.item())
    mean_mse_list.append(total_mse.item())
    temperature = start_temp * (min_temp / start_temp) ** (1 / (n_iter - 1))
    length_scale_list.append(length_scale.detach().cpu().numpy())

    if (k + 1) % 100 == 0:
        plt.figure(figsize=(10, 5))
        plt.title(f'Logits distribution at epoch {k + 1}')
        plt.imshow(1 - 1 / (1 + np.exp(-model_GS.logits[1, 0, :, :].detach().cpu().numpy())), cmap='viridis', interpolation='nearest', vmin=0, vmax=1)
        plt.colorbar()
        plt.show()
    #torch.cuda.empty_cache()
    length_scale_list.append(length_scale.detach().cpu().numpy())
    
# Plot mean loss
plt.figure(figsize=(10, 5))
plt.plot(mean_loss_list, label='Mean Model Loss')
plt.xlabel('Iteration')
plt.ylabel('Mean Model Loss')
plt.title('Mean Model Loss Over Iterations')
plt.legend()
plt.show()

# Plot mean normalized selected points
plt.figure(figsize=(10, 5))
plt.plot(mean_points_list, label='Mean Normalized Selected Points')
plt.xlabel('Iteration')
plt.ylabel('Mean Normalized Selected Points')
plt.title('Mean Normalized Selected Points Over Iterations')
plt.legend()
plt.show()

# Plot mean MSE loss
plt.figure(figsize=(10, 5))
plt.plot(mean_mse_list, label='Mean RMSE Loss')
plt.xlabel('Iteration')
plt.ylabel('Mean RMSE Loss')
plt.title('Mean RMSE Loss Over Iterations')
plt.legend()
plt.show()

# Plot mean MSE loss
plt.figure(figsize=(10, 5))
plt.plot(length_scale_list, label='Length scale')
plt.xlabel('Iteration')
plt.ylabel('Length scale')
plt.title('Length scale Over Iterations')
plt.legend()
plt.show()

# Visualization code (unchanged)
tensor_tgt = batch.tgt[0,0,0].detach().cpu().numpy()
da = xr.DataArray(tensor_tgt, dims=inp_da_GS.dims, coords=inp_da_GS.coords)
da = da * std_tgt + mean_tgt

plt.figure(figsize=(10, 5))
plt.title('Observation probability after training')
plt.imshow(1 - 1 / (1 + np.exp(-model_GS.logits[1, 0, :, :].detach().cpu().numpy())), cmap='viridis', interpolation='nearest', vmin=0, vmax=1)
plt.colorbar()
plt.show()

logits_init = (torch.zeros((2, time, lat, lon)))
logits_init.data[1, :, :, :] = np.log(rate / (1 - rate))
plt.figure(figsize=(10, 5))
plt.title('Observation probability before training')
plt.imshow(1 - 1 / (1 + np.exp(-logits_init[1, 0, :, :].detach().cpu().numpy())), cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.show()

plt.figure(figsize=(10, 5))
da.plot()
plt.title(f"Sample Values at Time Step {time_step}")
plt.show()

# Metrics

In [ ]:
logits_inits_init = torch.zeros((2,1,200,200)).to(device)
logits_inits_init[1, :, :, :] = np.log(rate / (1 - rate))

tau = 1
hard = True 
padding_size = 10  # Adjust this value as needed
inp_da_GS = inp_da.isel(lat=slice(crop_lat_start, crop_lat_end),
                     lon=slice(crop_lon_start, crop_lon_end))

mean_tgt = inp_da_GS.mean().item()
std_tgt = inp_da_GS.std().item()

tens_inp_da = torch.from_numpy(inp_da_GS.values).float().to(device)
tens_inp_da = (tens_inp_da - mean_tgt) / std_tgt
tens_inp_da = tens_inp_da.unsqueeze(0)
tens_inp_da = tens_inp_da.unsqueeze(0)
tens_inp_da.requires_grad_(True)
time= 1
lat, lon =  inp_da_GS.shape
tgt_inp = tens_inp_da.clone()

losses_init = []
exp_var_init = []

for _ in range(100):
    gs_output = F.gumbel_softmax(logits_inits_init, hard=True, dim=0)[0, :, :, :]
    mask_input = gs_output.view(time,lat, lon)
    mask_input = mask_input.unsqueeze(0)  # Add batch dimension if necessary
    mask = mask_input
    selected_tgt_inp = tgt_inp * mask
    selected_tgt_inp[mask == 0] = float('nan')
    # Perform optimal interpolation
    output = optimal_interpolation(selected_tgt_inp[0,0], length_scale=length_scale_list[0])

    # Plot the output tensor
    output_numpy = output.detach().cpu().numpy()
    tgt_center_numpy = batch.tgt.detach().cpu().numpy()

    output_numpy_non = (output_numpy * std_tgt) + mean_tgt
    tgt_center_non = (tgt_center_numpy * std_tgt) + mean_tgt

    rmse_loss = np.sqrt(np.nanmean((tgt_center_non - output_numpy_non) ** 2))

    #loss_mse = F.mse_loss(output_center, tgt_center[0,0])
    losses_init.append(rmse_loss)

    # Calculate variance of residuals and actual values
    residual_variance = np.var(output_numpy_non - tgt_center_non, ddof=1)
    actual_variance = np.var(tgt_center_non, ddof=1)

    # Explained Variance
    explained_variance = 1 - (residual_variance / actual_variance)
    exp_var_init.append(explained_variance)

# Plot histogram of the losses
plt.figure(figsize=(10, 5))
#plt.hist(losses_trained, bins=20, edgecolor='black', color='red')
plt.hist(losses_init, bins=20, edgecolor='black')
plt.xlabel('Loss')
plt.ylabel('Frequency')
plt.title('Histogram of RMSE Over 100 samples uniform mask')
plt.show()

# Plot histogram of the losses
plt.figure(figsize=(10, 5))
#plt.hist(losses_trained, bins=20, edgecolor='black', color='red')
plt.hist(exp_var_init, bins=20, edgecolor='black')
plt.xlabel('Explained variance')
plt.ylabel('Frequency')
plt.title('Histogram of the explained variance Over 100 samples uniform mask')
plt.show()

# Print the mean of the losses
mean_loss_value_init = np.mean(losses_init)
print(f"Mean Loss untrained: {mean_loss_value_init}")

gs_output = F.gumbel_softmax(logits_inits_init, tau=tau, hard=hard, dim=0)[0, :, :, :]
mask_input = gs_output.view(time,lat, lon)
mask_input = mask_input.unsqueeze(0).to(device)  # Add batch dimension if necessary
normalized_selected_points = mask_input.mean()

valid_points_mask = ~torch.isnan(selected_tgt_inp)
print(f"Normalized Selected Points: {normalized_selected_points}")

In [ ]:
logits = model_GS.logits

losses_trained = []
exp_var_trained = []
    
for _ in range(100):
    gs_output = F.gumbel_softmax(logits, hard=True, dim=0)[0, :, :, :]
    mask_input = gs_output.view(time,lat, lon)
    mask_input = mask_input.unsqueeze(0).to(device)  # Add batch dimension if necessary
    mask = mask_input
    selected_tgt_inp = tgt_inp * mask
    selected_tgt_inp[mask == 0] = float('nan')
    # Perform optimal interpolation
    output = optimal_interpolation(selected_tgt_inp[0,0], length_scale=length_scale_list[-1])

    # Plot the output tensor
    output_numpy = output.detach().cpu().numpy()
    tgt_center_numpy = batch.tgt.detach().cpu().numpy()

    output_numpy_non = (output_numpy * std_tgt) + mean_tgt
    tgt_center_non = (tgt_center_numpy * std_tgt) + mean_tgt

    rmse_loss = np.sqrt(np.nanmean((tgt_center_non - output_numpy_non) ** 2))

    #loss_mse = F.mse_loss(output_center, tgt_center[0,0])
    losses_trained.append(rmse_loss)

    # Calculate variance of residuals and actual values
    residual_variance = np.var(output_numpy_non - tgt_center_non, ddof=1)
    actual_variance = np.var(tgt_center_non, ddof=1)

    # Explained Variance
    explained_variance = 1 - (residual_variance / actual_variance)
    exp_var_trained.append(explained_variance)


# Plot histogram of the losses
plt.figure(figsize=(10, 5))
#plt.hist(losses_trained, bins=20, edgecolor='black', color='red')
plt.hist(losses_trained, bins=20, edgecolor='black', color='red')
plt.xlabel('Loss')
plt.ylabel('Frequency')
plt.title('Histogram of RMSE Over 100 samples trained warped mask')
plt.show()

# Plot histogram of the losses
plt.figure(figsize=(10, 5))
#plt.hist(losses_trained, bins=20, edgecolor='black', color='red')
plt.hist(exp_var_trained, bins=20, edgecolor='black', color='red')
plt.xlabel('Explained variance')
plt.ylabel('Frequency')
plt.title('Histogram of the explained variance Over 100 samples warped mask')
plt.show()

# Print the mean of the losses
mean_loss_value_trained= np.mean(losses_trained)
print(f"Mean Loss warped: {mean_loss_value_trained}")

# Generate mask using Gumbel-Softmax
gs_output_warped = F.gumbel_softmax(logits, tau=tau, hard=hard, dim=0)[0, :, :, :]
mask_input_warped = gs_output_warped.view(time,lat, lon)
mask_input_warped = mask_input_warped.unsqueeze(0)  # Add batch dimension if necessary
normalized_selected_points_warped = mask_input_warped.mean()
print(f"Normalized Selected Points: {normalized_selected_points_warped}")

In [ ]:
# Plot histogram of the losses
plt.figure(figsize=(10, 5))
plt.hist(losses_trained, bins=20, edgecolor='black', color='red')
plt.hist(losses_init, bins=20, edgecolor='black')
plt.xlabel('Loss')
plt.ylabel('Frequency')
plt.title('Histogram of RMSE Over 100 samples uniform mask in blue, trained mask in red and warped in green')
plt.show()

# Plot histogram of the losses
plt.figure(figsize=(10, 5))
plt.hist(exp_var_trained, bins=20, edgecolor='black', color='red')
plt.hist(exp_var_init, bins=20, edgecolor='black')
plt.xlabel('Explained variance')
plt.ylabel('Frequency')
plt.title('Histogram of the explained variance Over 100 samples uniform mask in blue, trained mask in red and warped in green')
plt.show()

# Count how many uniform samples have RMSE worse than the trained mean RMSE
worse_rmse_count = np.sum(np.array(losses_init) > mean_loss_value_trained)
better_rmse_count = np.sum(np.array(losses_init) < mean_loss_value_trained)
total_uniform_samples = len(losses_init)

percentage_worse_rmse = (worse_rmse_count / total_uniform_samples) * 100
percentage_better_rmse = (better_rmse_count / total_uniform_samples) * 100

print(f"Percentage of uniform RMSE values worse than the trained mean RMSE: {percentage_worse_rmse:.2f}%")
print(f"Percentage of uniform RMSE values better than the trained mean RMSE: {percentage_better_rmse:.2f}%")

# For explained variance, higher values are better.
mean_ev_trained = np.mean(exp_var_trained)
mean_ev_uniform = np.mean(exp_var_init)

# Count how many uniform samples have explained variance worse (i.e., lower) than the trained mean
worse_ev_count = np.sum(np.array(exp_var_init) < mean_ev_trained)
better_ev_count = np.sum(np.array(exp_var_init) > mean_ev_trained)
total_uniform_samples_ev = len(exp_var_init)

percentage_worse_ev = (worse_ev_count / total_uniform_samples_ev) * 100
percentage_better_ev = (better_ev_count / total_uniform_samples_ev) * 100

print(f"Percentage of uniform explained variance values worse than the trained mean EV: {percentage_worse_ev:.2f}%")
print(f"Percentage of uniform explained variance values better than the trained mean EV: {percentage_better_ev:.2f}%")